In [ ]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper

In [16]:
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb')
coefficients = cnn_mlp_encoder.get_hamiltonian(tensor)
qubit_instructions = jw_quantum_mapper.apply_jw(coefficients)

tensor.shape
coefficients
qubit_instructions

['0.0372 * Z0',
 '-0.0237 * Z1',
 '0.0182 * Z2',
 '0.0128 * Z3',
 '-0.0355 * X0 X1',
 '-0.0355 * Y0 Y1',
 '-0.0168 * X0 Z1 X2',
 '-0.0168 * Y0 Z1 Y2',
 '-0.0294 * X0 Z1 Z2 X3',
 '-0.0294 * Y0 Z1 Z2 Y3',
 '0.0047 * X1 X2',
 '0.0047 * Y1 Y2',
 '-0.0425 * X1 Z2 X3',
 '-0.0425 * Y1 Z2 Y3',
 '0.0079 * X2 X3',
 '0.0079 * Y2 Y3']

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

In [ ]:
def visualize_filters(layer):
# get learned filter weights 
# detach = removes weights from computation graph, so it doesn't compute gradients only weights
  filters = layer.weight.detach().cpu()
# shape (out channels, in channels, depth, height, width), so only takes out channels needed
  num_filters = filters.shape[0]
# creates figure, just general size 12x8
  plt.figure(figsize = (12, 8))
# loop to go through every filter 
  for i in range(num_filters): 
    # get one filter, so first input channel 
    filter = filters[i, 0]
    # show middle slide of 3d filter, 
    middle_slice = filter[1]
    # create grid of plots 
    plt.subplot(4,4, i + 1)
    # convert numbers to pixels and use grayscale so positive weights = lighter vs negative = darker
    plt.imshow(middle_slice, cmap = "gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.show()

In [ ]:
def visualize_feature_maps(layer, image):
    # Get feature maps from the convolution layer
    # makes it so model doesnt calculate gradients 
    with torch.no_grad():
        feature_maps = layer(image).cpu()
    # Remove batch dimension, so batchsize (batchsize, channels, dxhxw)
    feature_maps = feature_maps.squeeze(0)
    # Plot each feature map
    plt.figure(figsize=(12,8))
    for i in range(feature_maps.shape[0]):
        # Get one feature map
        fmap = feature_maps[i]
        # Take middle slice of 3D feature map, so only shows a 2d slice 
        middle_slice = fmap[fmap.shape[0]//2]
        # creates grid location, 4 rows, 4 colums, teh current position
        plt.subplot(4,4,i+1)
        # colors images, adds title, and removes axis
        plt.imshow(middle_slice, cmap="gray")
        plt.title(f"Map {i+1}")
        plt.axis("off")

    plt.show()